## Concept 3 — Summary Memory

### What is it?
Instead of dropping old messages, an LLM **compresses the full conversation into a running summary**.
The summary is stored as a `SystemMessage` — compact but information-preserving across all turns.

### Pattern
```
After each turn:
  existing_summary + new_turn → LLM → updated_summary (1-3 sentences)

Prompt structure:
  [SystemMessage: 'Conversation so far: {summary}'] + [HumanMessage: {query}]
```

### Limitation (what Concept 4 fixes)
Summarization is lossy — exact wording and numbers can be lost. Also adds an LLM call per turn.
→ Concept 4 keeps recent turns raw and only summarizes older ones.

In [ ]:
print('hi')

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

from MemoryUtils import get_llm, TEST_TURNS, make_history, load_test_history, run_recall_test
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = get_llm()
print('LLM ready')

### Step 1 — The Summarization Function
Two modes: create a fresh summary, or update an existing one with a new turn.

In [ ]:
def update_summary(current_summary: str, human: str, ai: str) -> str:
    """Update the running summary with a new human+AI turn."""
    if not current_summary:
        prompt = f'Summarize this exchange in 1-2 sentences:\nHuman: {human}\nAI: {ai}'
    else:
        prompt = (
            f'Update the conversation summary with the new exchange.\n'
            f'Current summary: {current_summary}\n'
            f'New exchange - Human: {human} / AI: {ai}\n'
            f'Write an updated summary in 1-3 sentences, keeping all key facts.'
        )
    return llm.invoke(prompt).content

# Quick test
test_summary = update_summary('', 'My name is Mohan.', 'Hello Mohan!')
print('Test summary:', test_summary)

### Step 2 — Watch Summary Grow Across All 5 Turns
The summary accumulates all facts while staying compact.

In [ ]:
running_summary = ''

for i, (human, ai) in enumerate(TEST_TURNS):
    running_summary = update_summary(running_summary, human, ai)
    print(f'After Turn {i+1} (~{len(running_summary)//4} tokens):')
    print(f'  {running_summary}')
    print()

### Step 3 — Compare Size: Buffer vs Summary

In [ ]:
history = make_history()
load_test_history(history)

buf_tokens = sum(len(m.content) for m in history.messages) // 4
sum_tokens = len(running_summary) // 4

print('After 5 turns:')
print(f'  Buffer Memory  : {len(history.messages)} messages  ~{buf_tokens} tokens')
print(f'  Summary Memory : 1 string          ~{sum_tokens} tokens')
print()
print('Summary tokens stays roughly constant no matter how long the conversation gets.')

### Full Function — chat_with_summary
Injects the running summary as a SystemMessage. Updates summary after each response.

In [ ]:
# Build the final running summary from all TEST_TURNS
final_summary = ''
for human, ai in TEST_TURNS:
    final_summary = update_summary(final_summary, human, ai)

print('Summary to be used:')
print(final_summary)

def chat_with_summary(query):
    global final_summary
    messages = [
        SystemMessage(content=f'You are a helpful assistant. Conversation so far: {final_summary}'),
        HumanMessage(content=query),
    ]
    response = llm.invoke(messages)
    # Update summary with this new turn
    final_summary = update_summary(final_summary, query, response.content)
    return response.content

### Test — Recall via Summary
All 5 facts are in the summary. Recall should work, but answers may be less precise than buffer.

In [ ]:
run_recall_test(chat_with_summary, label='Concept 3 -- Summary Memory')